# NOTEBOOK 03 – LAM LINH | CHƯƠNG 3: DỮ LIỆU VÀ EDA

**Phụ trách:** 3.1–3.5 · Issue **#4**; phối hợp kiểm soát leakage Issue **#8**.
**Bàn giao schema/dataset cơ sở:** 12/11/2026. **Báo cáo Ch3:** 16/11/2026.

**Đầu vào:** 9 CSV Olist trong `data/raw/`; tải hợp lệ từ [Kaggle](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce).
**Đầu ra dùng chung:** `data/processed/orders_model_ready.csv` (không commit); kiểm tra QA + danh sách feature tại mốc duyệt.

[Lý thuyết Ch3](../docs/theory/03_lam_linh_ch3.md) · [DATA_CONTRACT](../docs/DATA_CONTRACT.md).

> Các cell có thể chạy khi dữ liệu Olist đã nằm trong data/raw. Không khẳng định đã xử lý xong hoặc gán kết quả khi chưa chạy.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def find_project_root():
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "docs" / "DATA_CONTRACT.md").exists():
            return candidate
    raise FileNotFoundError("Mở notebook trong repository (hoặc thư mục notebooks/).")
ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)
print("Project root:", ROOT)

## 3.1 Nguồn dữ liệu, cấu trúc và đọc file

**Nội dung báo cáo:** nguồn dataset, bản chất giao dịch Olist Brazil 2016–2018, 9 bảng và các khóa (`order_id`, `customer_id`, `product_id`, `seller_id`). Chỉ tóm tắt số dòng/biến **sau khi tự chạy** cell kiểm kê.

**Ràng buộc:** dữ liệu gốc không đưa lên GitHub.

In [ ]:
names = [
    "olist_orders_dataset.csv", "olist_order_items_dataset.csv",
    "olist_customers_dataset.csv", "olist_sellers_dataset.csv",
    "olist_products_dataset.csv", "olist_order_payments_dataset.csv",
    "olist_order_reviews_dataset.csv", "olist_geolocation_dataset.csv",
    "product_category_name_translation.csv",
]
missing = [name for name in names if not (RAW / name).exists()]
if missing:
    raise FileNotFoundError(
        "Thiếu các tệp Olist trong data/raw/: " + ", ".join(missing)
        + ". Tải từ Kaggle theo data/README.md."
    )
tables = {name.replace(".csv", ""): pd.read_csv(RAW / name) for name in names}
profile = pd.DataFrame([
    {"table": name, "rows": len(df), "columns": df.shape[1],
     "duplicate_rows": int(df.duplicated().sum()), "missing_cells": int(df.isna().sum().sum())}
    for name, df in tables.items()
])
display(profile)

## 3.2 Tích hợp và làm sạch dữ liệu

**Rủi ro quan trọng:** `order_items` có nhiều dòng/order, `geolocation` có thể nhiều dòng/zip prefix. **Không join trực tiếp để một đơn thành nhiều dòng rồi tính KPI sai.**

Trong phiên bản khung này, chúng ta tổng hợp `order_items` theo `order_id`, sau đó merge với bảng `orders`, `customers`. Các dữ liệu sản phẩm/seller phục vụ **EDA và feature ứng viên**, chưa tự động được sử dụng để dự báo sớm nếu thiếu bằng chứng biết được tại `order_approved_at`.

In [ ]:
from src.contracts import assert_unique_orders
from src.labels import make_late_label

orders = tables["olist_orders_dataset"].copy()
items = tables["olist_order_items_dataset"].copy()
customers = tables["olist_customers_dataset"].copy()
sellers = tables["olist_sellers_dataset"].copy()
products = tables["olist_products_dataset"].copy()

assert_unique_orders(orders)
assert not customers["customer_id"].duplicated().any()
assert not products["product_id"].duplicated().any()
assert not sellers["seller_id"].duplicated().any()

items_enriched = (
    items.merge(products[["product_id", "product_category_name", "product_weight_g"]],
                on="product_id", how="left", validate="many_to_one")
         .merge(sellers[["seller_id", "seller_state"]],
                on="seller_id", how="left", validate="many_to_one")
)
def summarize_distinct(series):
    vals = sorted(series.dropna().astype(str).unique())
    return vals[0] if len(vals) == 1 else ("MULTI" if len(vals) > 1 else "UNKNOWN")

item_agg = items_enriched.groupby("order_id", as_index=False).agg(
    item_count=("product_id", "size"),
    product_count=("product_id", "nunique"),
    seller_count=("seller_id", "nunique"),
    total_price=("price", "sum"),
    total_freight=("freight_value", "sum"),
    avg_weight_g=("product_weight_g", "mean"),
    seller_region=("seller_state", summarize_distinct),
    product_category=("product_category_name", summarize_distinct),
)
base = (
    orders.merge(customers[["customer_id", "customer_state"]],
                 on="customer_id", how="left", validate="many_to_one")
          .merge(item_agg, on="order_id", how="left", validate="one_to_one")
)
assert_unique_orders(base)
print("Orders:", len(orders), "| Orders after merge:", len(base))
display(base[["order_id", "customer_state", "item_count", "seller_count"]].head())

## 3.3 Thiết lập nhãn và chọn mẫu

**Nhãn:** `is_late=1` khi `DATE(delivered_customer) > DATE(estimated_delivery)`. Chỉ những đơn *delivered*, có phê duyệt và 2 ngày hợp lệ mới có nhãn 0/1.

**Không gán nhãn 0 cho cancelled/unavailable**, ghi rõ số mẫu bị loại và tác động delivered-only selection bias.

In [ ]:
date_columns = [
    "order_purchase_timestamp", "order_approved_at",
    "order_delivered_carrier_date", "order_delivered_customer_date",
    "order_estimated_delivery_date"
]
for col in date_columns:
    base[col] = pd.to_datetime(base[col], errors="coerce")

base["is_late"] = make_late_label(base)
label_audit = pd.DataFrame({
    "metric": ["all_orders", "labeled_orders", "unlabeled_orders", "late_orders", "on_time_orders"],
    "count": [
        len(base), int(base["is_late"].notna().sum()), int(base["is_late"].isna().sum()),
        int((base["is_late"] == 1).sum()), int((base["is_late"] == 0).sum())
    ]
})
display(label_audit)
model_data = base.loc[base["is_late"].notna()].copy()
model_data["is_late"] = model_data["is_late"].astype(int)
assert_unique_orders(model_data)

## 3.4 EDA: thống kê và trực quan hóa có diễn giải

Đề xuất EDA thực tế: phân phối số đơn theo tháng, tỷ lệ giao trễ theo bang người mua, theo vùng seller (MULTI tách riêng), nhóm hàng và trọng lượng. **Báo cáo luôn số lượng đơn cùng tỷ lệ**, tránh suy luận nhóm nhỏ đại diện cả ngành.

Viết 3–5 insight sau khi đọc biểu đồ; không tự diễn giải trước khi chạy.

In [ ]:
model_data["purchase_month"] = model_data["order_purchase_timestamp"].dt.to_period("M").astype(str)
monthly = model_data.groupby("purchase_month", as_index=False).agg(
    orders=("order_id", "nunique"),
    late_rate=("is_late", "mean")
)
by_state = model_data.groupby("customer_state", dropna=False, as_index=False).agg(
    orders=("order_id", "nunique"),
    late_rate=("is_late", "mean")
).sort_values("orders", ascending=False)
display(monthly.tail(12))
display(by_state.head(15))
ax = monthly.plot(x="purchase_month", y="late_rate", kind="line", figsize=(10,4), marker="o")
ax.set(xlabel="Tháng đặt hàng", ylabel="Tỷ lệ giao trễ", title="Tỷ lệ giao trễ theo tháng (mẫu delivered)")
plt.xticks(rotation=60)
plt.tight_layout()
plt.show()

## 3.5 Feature engineering & leakage audit

Tập **an toàn tối thiểu** cho bài toán dự báo tại `order_approved_at`: tháng đặt hàng, ngày trong tuần, giờ đặt, thời gian từ đặt đến phê duyệt, bang khách. Các feature về giá/phí, estimated date, product/seller cần **xác minh thời điểm xuất hiện trước khi đưa vào model**.

Các cột `order_delivered_customer_date`, `order_delivered_carrier_date`, `order_status` cuối và `review_score` bị cấm làm đặc trưng.

Giữ thời điểm giao thực tế trong *artifact nội bộ* chỉ để đánh giá tính trưởng thành nhãn ở Ch4; không đưa cột này vào model X.

In [ ]:
model_data["purchase_month_number"] = model_data["order_purchase_timestamp"].dt.month
model_data["purchase_weekday"] = model_data["order_purchase_timestamp"].dt.weekday
model_data["purchase_hour"] = model_data["order_purchase_timestamp"].dt.hour
model_data["approval_delay_hours"] = (
    (model_data["order_approved_at"] - model_data["order_purchase_timestamp"])
    .dt.total_seconds() / 3600
)
# Baseline an toàn: chỉ dùng các feature có sẵn trước/lúc đơn được duyệt.
SAFE_FEATURES = [
    "purchase_month_number", "purchase_weekday",
    "purchase_hour", "approval_delay_hours", "customer_state"
]
TARGET = "is_late"
META = ["order_id", "order_approved_at", "order_delivered_customer_date", TARGET]
assert set(SAFE_FEATURES).issubset(model_data.columns)
assert not model_data[TARGET].isna().any()
assert_unique_orders(model_data)
output = PROCESSED / "orders_model_ready.csv"
model_data.to_csv(output, index=False)
print("Đã xuất:", output, "| Số đơn có nhãn:", len(model_data))
print("SAFE_FEATURES dùng cho mô hình baseline:", SAFE_FEATURES)

## 3.6 Bàn giao và hướng dẫn viết báo cáo
- [ ] Đã tải 9 CSV và lập bảng row/column/missing/duplicates.
- [ ] Đã tổng hợp trước khi join; `assert_unique_orders` thành công.
- [ ] Đã thống kê đơn không gán nhãn, nêu rõ lý do loại.
- [ ] EDA có ít nhất biểu đồ theo tháng, bang, seller/danh mục và nhận xét sau khi chạy.
- [ ] Tạo file `orders_model_ready.csv` + ghi schema, số mẫu, danh sách feature, CSV hash/version.
- [ ] Gửi cho Trúc Chi và Vĩnh Chi, xác nhận đã nhận trong Issue #4.
- [ ] Viết 3.1–3.5 trong `reports/chapters/CH03_DATA.md` với hình bảng đánh số.

**Đầu ra chỉ mang tính nghiên cứu trên tập delivered đã có nhãn.** Phải ghi rõ selection bias và không tự giới thiệu feature ứng viên là feature hợp lệ khi chưa xác minh.